# Week 28, Day 4: Measuring activity I: electrodes, from single units to Neuropixels

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w028/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fs, T = 30000, 60.0
n = int(fs * T); t = np.arange(n) / fs
rs = np.random.default_rng(28)

def spike_train(rate, ref=0.002):
    isi = ref + rs.exponential(1 / rate, int(rate * T * 2))
    st = np.cumsum(isi); return st[st < T - 0.01]

wt = np.arange(-30, 45) / fs                     # 2.5 ms of waveform
shapeA = -np.exp(-(wt / 0.0002) ** 2) + 0.35 * np.exp(-((wt - 0.0006) / 0.00035) ** 2)
shapeB = -np.exp(-(wt / 0.0003) ** 2) + 0.20 * np.exp(-((wt - 0.0009) / 0.0005) ** 2)
trains = {"A": spike_train(10.0), "B": spike_train(10.0)}
x = 3.0 * np.sin(2 * np.pi * 6 * t) + 1.0 * rs.normal(size=n)       # slow LFP plus noise (arbitrary units)
for name, shape, amp in (("A", shapeA, 9.0), ("B", shapeB, 6.5)):
    for s in trains[name]:
        i = int(s * fs); x[i - 30:i + 45] += amp * shape
print(f"{len(trains['A'])} spikes from A, {len(trains['B'])} from B in {T:.0f} s")

### Exercise: Finding the spikes

High-pass filter the trace by zeroing all FFT components below 300 Hz to get `xf`. Set the threshold `thr` at −4.5 times the robust noise estimate, and detect a spike at every downward crossing (ignoring crossings within 1 ms of the previous one), keeping the sample of the minimum within the next 1 ms, in `det` (sample indices). Then compute the fraction of true spikes detected within 0.5 ms (`hit_rate`) and the fraction of detections matching no true spike (`false_rate`).

In [ ]:
X = np.fft.rfft(x); f = np.fft.rfftfreq(n, 1 / fs)
X[f < 300] = 0
xf = ...
thr = ...
cross = np.where((xf[1:] < thr) & (xf[:-1] >= thr))[0] + 1
det, last = [], -10**9
for c in cross:                                    # 1 ms dead time: one detection per spike
    if c - last > 30:
        det.append(c + np.argmin(xf[c:c + 30])); last = c
det = np.array(det)
true = np.sort(np.concatenate([trains["A"], trains["B"]]) * fs).astype(int)
near = lambda a, b: np.min(np.abs(b[:, None] - a[None, :]), axis=1) <= 15   # for each of b, is some a within 0.5 ms?
hit_rate = ...
false_rate = ...
print(f"threshold {thr:.2f};  {len(det)} detections;  hits {hit_rate:.3f}, false {false_rate:.3f}")

In [ ]:
import numpy as np
def test_filter():
    "xf has the 6 Hz LFP removed: its power at 6 Hz is below 1% of the raw trace's"
    f = np.fft.rfftfreq(n, 1 / fs); k = np.argmin(np.abs(f - 6))
    assert np.abs(np.fft.rfft(xf))[k] < 0.01 * np.abs(np.fft.rfft(x))[k]

def test_threshold():
    "thr is -4.5 times the median absolute deviation estimate of the noise"
    assert abs(thr + 4.5 * np.median(np.abs(xf)) / 0.6745) < 1e-9

def test_hits():
    "over 90% of true spikes are detected within 0.5 ms, with under 5% false detections"
    assert hit_rate > 0.9 and false_rate < 0.05

test_filter()
test_threshold()
test_hits()
print("All checks passed")

### Exercise: PCA and clustering of waveforms

Cut a 75-sample snippet around each detection (30 before, 45 after), take the top two principal components, and cluster them into two units with k-means (10 restarts, local rng seed 2). For detections that match a true spike, compare the units with the true neuron, taking the better naming, and store the result in `sort_accuracy`.

In [ ]:
det = det[(det > 30) & (det < n - 45)]
snips = np.array([xf[d - 30:d + 45] for d in det])
S = snips - snips.mean(0)
Vt = np.linalg.svd(S, full_matrices=False)[2]
F = S @ Vt[:2].T

def kmeans(X, k=2, iters=50, restarts=10):
    best, best_cost, rk = None, np.inf, np.random.default_rng(2)
    for _ in range(restarts):
        c = X[rk.choice(len(X), k, replace=False)]
        for _ in range(iters):
            d = ((X[:, None] - c[None]) ** 2).sum(-1); lab = d.argmin(1)
            c = np.array([X[lab == j].mean(0) if np.any(lab == j) else c[j] for j in range(k)])
        cost = d[np.arange(len(X)), lab].sum()
        if cost < best_cost: best, best_cost = lab, cost
    return best

units = kmeans(F)
tA, tB = (trains["A"] * fs).astype(int), (trains["B"] * fs).astype(int)
isA = np.min(np.abs(det[:, None] - tA[None]), 1) <= 15
isB = np.min(np.abs(det[:, None] - tB[None]), 1) <= 15
ok = isA ^ isB
sort_accuracy = ...
print(f"sorting accuracy {sort_accuracy:.3f} on {ok.sum()} matched detections")
plt.scatter(F[:, 0], F[:, 1], c=units, s=4); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()

In [ ]:
import numpy as np
def test_snips():
    "snips holds one 2.5 ms waveform (75 samples) per detection"
    assert snips.shape == (len(det), 75)

def test_accuracy():
    "the sorted units match the true neuron of each matched detection with over 95% accuracy"
    assert sort_accuracy > 0.95

test_snips()
test_accuracy()
print("All checks passed")

### Exercise: Refractory violations

Write `violations(times)`, the fraction of intervals between consecutive sorted spike times (in seconds) shorter than 2 ms. Compute it for each sorted unit (`v_units`, a list of two) and for all detections pooled into one unit (`v_merged`).

In [ ]:
def violations(times):
    isi = np.diff(np.sort(times))
    return ...

det_t = det / fs
v_units = [violations(det_t[units == u]) for u in (0, 1)]
v_merged = ...
print(f"violations: units {np.round(v_units, 4)}, merged {v_merged:.4f}")

In [ ]:
import numpy as np
def test_function():
    "violations returns the fraction of interspike intervals below 2 ms"
    assert abs(violations(np.array([0.0, 0.001, 0.010, 0.011])) - 2 / 3) < 1e-12

def test_merged():
    "the merged unit violates far more often than either sorted unit"
    assert v_merged > 3 * max(v_units)

test_function()
test_merged()
print("All checks passed")